In [ ]:
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import spacy
from dotenv import load_dotenv
from hdbscan import HDBSCAN
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

from tqdm import tqdm
from umap import UMAP

from src.orchestrator import LLMOrchestrator
from src.analytics import AnalyticsModule
from src.data_loader import DataLoader

warnings.filterwarnings('ignore')
load_dotenv()

In [ ]:
# Sentiment Analysis with HerBERT (standalone)
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "Voicelab/herbert-base-cased-sentiment"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
access_token = os.environ.get("HF_TOKEN")

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)
model = AutoModelForSequenceClassification.from_pretrained(model_id,
    token=access_token,
).to(device)

text = "Jako do ceny dobra, ale nie jest to produkt wysokiej jakości. Szybko się zepsuł i nie spełnił moich oczekiwań."

encoding = tokenizer(
    text,
    add_special_tokens=True,
    truncation=True,
    padding="longest",
    return_attention_mask=True,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    logits = model(**encoding).logits

output = logits.to("cpu").numpy()
prediction = id2label[int(np.argmax(output))]
print(f"{text} ---> {prediction}")

In [ ]:
df_allegro_reviews = pd.read_csv('data/allegro_ground_truth.csv', encoding='utf-8-sig',sep=";")
df_allegro_reviews.groupby("kategoria_benchmark").size().sort_values(ascending=False) 

In [ ]:
# Inicjalizacja loadera i przygotowanie danych
data_loader = DataLoader()
raw_texts, cleaned_texts, texts, all_stopwords = data_loader.prepare_texts(
    df_allegro_reviews, 
    text_column="text", 
    min_word_length=3
)

In [ ]:
import pandas as pd
import numpy as np
import itertools
from IPython.display import display
import time
model_roberta = SentenceTransformer("sdadas/mmlw-roberta-base")
vectorizer_model = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 2))
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

# 1. Obliczenie embeddingów
print("Obliczanie wektorów z modelu RoBERTa...")
embeddings = model_roberta.encode(texts, show_progress_bar=True)

upper_limit = int(round(len(texts)/10,0))
# 2. Parametry do przetestowania (możesz dostosować wartości)
# umap_n_neighbors = [5, 10, 15, 20, 25]
umap_n_neighbors = [i for i in range(5, upper_limit, 5)]
# hdbscan_min_cluster_sizes = [5, 10, 20, 25, 30]
hdbscan_min_cluster_sizes = [i for i in range(5, upper_limit, 5)]
results = []

# 3. Pętla testująca każdą kombinację
for n_neighbors, min_cluster_size in itertools.product(umap_n_neighbors, hdbscan_min_cluster_sizes):
    start_time = time.time()
    # Inicjalizacja modeli z nowymi parametrami
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=2, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    
    # Inicjalizacja BERTopic
    topic_model = BERTopic(
        embedding_model=model_roberta,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model = ClassTfidfTransformer(),
        representation_model=representation_model,
        nr_topics="auto" 
    )
    
    # Przekazujemy gotowe wektory w parametrze 'embeddings'
    topics, _ = topic_model.fit_transform(texts, embeddings=embeddings)
    
    # Obliczanie metryk
    topics_array = np.array(topics)
    noise_count = (topics_array == -1).sum()
    noise_pct = (noise_count / len(topics_array)) * 100
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    
    # Liczba rekordów w największym temacie (poza szumem)
    topic_counts = pd.Series(topics_array).value_counts()
    max_topic_size = topic_counts.drop(-1, errors='ignore').max() if n_topics > 0 else 0
    
    # Próba pobrania sensowności (top 3 słowa z największego klastra '0')
    top_words = "Brak klastrów"
    if n_topics > 0 and 0 in topic_model.get_topics():
        top_words = ", ".join([word for word, _ in topic_model.get_topic(0)[:3]])
    
    elapsed_time = time.time() - start_time
    # Zapis
    results.append({
        "UMAP_neighbors": n_neighbors,
        "HDBSCAN_min_size": min_cluster_size,
        "Liczba tematów": n_topics,
        "Szum (%)": round(noise_pct, 1),
        "Sensowność (Top słowa klastra 0)": top_words,
        "Max temat size": max_topic_size,
        "Czas (s)": round(elapsed_time, 2)
    })

# 4. Wyświetlenie wyników w formie tabeli
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by=["Szum (%)", "Liczba tematów"])
print("\n--- Wyniki eksperymentów z parametrami ---")
display(df_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Przekształcenie tabeli z wynikami do formatu macierzy (pivot)
pivot_noise = df_results.pivot(index="UMAP_neighbors", columns="HDBSCAN_min_size", values="Szum (%)")
pivot_topics = df_results.pivot(index="UMAP_neighbors", columns="HDBSCAN_min_size", values="Liczba tematów")

# Przygotowanie miejsca na 2 wykresy obok siebie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wykres 1: Udział szumu (%)
sns.heatmap(pivot_noise, annot=True, cmap="Reds", ax=axes[0], fmt=".1f", cbar_kws={'label': '% szumu'})
axes[0].set_title("Wpływ parametrów na wielkość szumu (%)")

# Wykres 2: Liczba tematów
sns.heatmap(pivot_topics, annot=True, cmap="Greens", ax=axes[1], fmt=".0f", cbar_kws={'label': 'liczba klastrów'})
axes[1].set_title("Wpływ parametrów na liczbę tematów")

plt.tight_layout()
plt.show()

# Wykres decyzyjny: Szum vs Liczba tematów (z automatycznym wykluczeniem outlierów)
# Identyfikacja outlierów dla obu osi (IQR method)
Q1_x = df_results["Szum (%)"].quantile(0.25)
Q3_x = df_results["Szum (%)"].quantile(0.75)    
IQR_x = Q3_x - Q1_x
lower_x = Q1_x - 1.5 * IQR_x
upper_x = Q3_x + 1.5 * IQR_x

Q1_y = df_results["Liczba tematów"].quantile(0.25)
Q3_y = df_results["Liczba tematów"].quantile(0.75)
IQR_y = Q3_y - Q1_y
lower_y = Q1_y - 1.5 * IQR_y
upper_y = Q3_y + 1.5 * IQR_y
max_topics_acceptable = int(max(df_results["Liczba tematów"])/2)+1
# Filtracja danych bez outlierów
df_filtered = df_results[
    (df_results["Szum (%)"] >= lower_x) & 
    (df_results["Szum (%)"] <= upper_x) & 
    (df_results["Liczba tematów"] >= lower_y) & 
    (df_results["Liczba tematów"] <= upper_y)
]

plt.figure(figsize=(10,7))
sns.scatterplot(data=df_filtered, x="Szum (%)", y="Liczba tematów", s=100, alpha=0.6)

# Dodaj etykiety dla każdego punktu
for idx, row in df_filtered.iterrows():
    plt.annotate(
        f"u:{int(row['UMAP_neighbors'])} h:{int(row['HDBSCAN_min_size'])}", 
        xy=(row['Szum (%)'], row['Liczba tematów']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=7,
        alpha=0.75
    )

plt.axvline(x=10, color='red', linestyle='--', linewidth=2, label='próg 10% szumu')
plt.axhline(y=max_topics_acceptable, color='green', linestyle='--', linewidth=2, label=f'próg {max_topics_acceptable} tematów')
plt.title(f"Szum (%) vs Liczba tematów (bez wartości skrajnych: {len(df_filtered)}/{len(df_results)} punktów)")
plt.xlabel("Szum (%)")
plt.ylabel("Liczba tematów")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Wyświetl tabelę parametrów w pełnej postaci
print("\n--- Parametry wszystkich punktów na wykresie ---")
params_table = df_filtered[['Szum (%)', 'Liczba tematów', 'UMAP_neighbors', 'HDBSCAN_min_size']].sort_values(by=['UMAP_neighbors', 'HDBSCAN_min_size'])
for idx, row in params_table.iterrows():
    print(f"u:{int(row['UMAP_neighbors']):2d} h:{int(row['HDBSCAN_min_size']):2d}  →  Szum: {row['Szum (%)']:5.1f}%  Tematy: {int(row['Liczba tematów']):2d}")

print(f"\nOdfiltrowano {len(df_results) - len(df_filtered)} wartości skrajnych.")

In [ ]:
# 2. INICJALIZACJA MODELI - HerBERT jako model embeddingowy, UMAP do redukcji wymiarów, HDBSCAN do klasteryzacji, KeyBERT do reprezentacji klastrów
model_roberta = SentenceTransformer("sdadas/mmlw-roberta-base")
# KROK 1: UMAP (Przygotowuje grunt dla HDBSCAN - stabilizuje klastry)
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
# KROK 2: HDBSCAN (odkrywa klastry - grupy podobnych opinii)
hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=2, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 2))
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

# 3. URUCHOMIENIE BERTopic
topic_model = BERTopic(
    embedding_model=model_roberta,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model = ClassTfidfTransformer(),
    representation_model=representation_model,
    nr_topics="auto"
)

topics, probs = topic_model.fit_transform(texts)

# 4. GET TOPIC INFORMATION
topic_info = topic_model.get_topic_info()
print("--- Klastry znalezione za pomocą modelu RoBERTa ---")
print(topic_info[["Topic", "Count", "Name"]].head(15))

# 5. QUALITY METRICS
topics_array = np.array(topics)
noise_count = (topics_array == -1).sum()
print(f"\nTopics found: {len(set(topics))}")
print(f"Szum stanowi: {noise_count / len(topics_array) * 100:.1f}%")
if len(set(topics)) > 1:
    print(f"Średnia wielkość klastra to: {len(topics_array) / (len(set(topics)) - (1 if -1 in topics else 0)):.0f}")
print("\nRozkład tematów:")
print(topic_info[["Topic", "Count"]].sort_values("Count", ascending=False))
topic_model.get_topic_info()
# topic_model.get_document_info(texts)

In [ ]:
topic_model.get_topic_info()
# topic_model.get_document_info(texts)

In [ ]:
#6. POST-PROCESSING: BUSINESS LABELS VIA GEMINI
orchestrator = LLMOrchestrator(mode="gemini", gemini_model="gemini-2.5-flash")

business_categories = [
    "Słabe materiały i usterki", "Kompatybilność i montaż", 
    "Zawyżone parametry i podróbki", "Niska wydajność w praktyce", 
    "Logistyka i dostawa", "Obsługa klienta", 
    "Pozytywna ocena funkcjonalności", "Szum informacyjny"
]

# Logi dla każdego topicu
labeling_results = []

def label_topic_with_gemini(row):
    topic_id = row['Topic']
    keywords = row['Representation']
    
    if topic_id == -1:
        labeling_results.append({
            'topic_id': topic_id,
            'kategoria': "Szum informacyjny",
            'prompt_tokens': 0,
            'completion_tokens': 0,
            'total_tokens': 0,
            'execution_time': 0.0
        })
        return "Szum informacyjny"
    
    start_time = time.time()
    docs = topic_model.get_representative_docs(topic_id)
    docs_context = "\n".join([f"- {doc[:150]}" for doc in docs[:3]])
    
    prompt = f"""Na podstawie poniższych słów kluczowych oraz przykładowych opinii klientów, 
przypisz tę grupę do DOKŁADNIE JEDNEJ z poniższych kategorii biznesowych:
{', '.join(business_categories)}

Słowa kluczowe z klastra: {', '.join(keywords[:5])}

Przykładowe opinie:
{docs_context}

Zwróć TYLKO nazwę kategorii, bez żadnych znaków interpunkcyjnych i komentarzy.
"""
    
    try:
        result, token_stats = orchestrator._call_gemini_with_usage(prompt)
        result = result.strip()
        execution_time = time.time() - start_time
        time.sleep(1)
        
        if result not in business_categories:
            kategoria = "Szum informacyjny"
        else:
            kategoria = result
            
        labeling_results.append({
            'topic_id': topic_id,
            'kategoria': kategoria,
            'prompt_tokens': token_stats.get('prompt_tokens', 0),
            'completion_tokens': token_stats.get('completion_tokens', 0),
            'total_tokens': token_stats.get('total_tokens', 0),
            'execution_time': execution_time
        })
        
        return kategoria
    except Exception as e:
        execution_time = time.time() - start_time
        print(f"Błąd dla klastra {topic_id}: {e}")
        labeling_results.append({
            'topic_id': topic_id,
            'kategoria': "Błąd API",
            'prompt_tokens': 0,
            'completion_tokens': 0,
            'total_tokens': 0,
            'execution_time': execution_time
        })
        return "Błąd API"

# Apply labeling z progress barem
tqdm.pandas(desc="Labeling topics with Gemini")
topic_info['kategoria_roberta_llm'] = topic_info.progress_apply(label_topic_with_gemini, axis=1)

# Utwórz DataFrame z logami
df_labeling_logs = pd.DataFrame(labeling_results)

# Połącz logi z topic_info
topic_info = topic_info.merge(df_labeling_logs, left_on='Topic', right_on='topic_id', how='left')

print("\n--- Wyniki po opisaniu przez LLM (z metrykami) ---")
print(topic_info[["Topic", "kategoria_roberta_llm", "Count", "prompt_tokens", "completion_tokens", "total_tokens", "execution_time"]].sort_values("Count", ascending=False))

# Podsumowanie
total_tokens_gemini_labeling = df_labeling_logs['total_tokens'].sum()
total_time_gemini_labeling = df_labeling_logs['execution_time'].sum()
print(f"\n--- Podsumowanie etykietowania ---")
print(f"Łączne tokeny: {total_tokens_gemini_labeling}")
print(f"Łączny czas: {total_time_gemini_labeling:.2f}s")
print(f"Średni czas per topic: {total_time_gemini_labeling/len(df_labeling_logs):.2f}s")

In [ ]:
# 7. CREATE ENRICHED RESULTS DATAFRAME FOR VISUALIZATION
# Create mappings from topic_id to labels and keywords
topic_to_label = dict(zip(topic_info["Topic"], topic_info["kategoria_roberta_llm"]))
topic_to_keywords = dict(zip(topic_info["Topic"], topic_info["Representation"]))

# Build results dataframe
df_roberta_llm = pd.DataFrame({
    "text": raw_texts[:len(texts)],  # Keep track of originals
    "text_cleaned": texts,
    "topic_id": topics,
    "confidence": probs,  # Confidence scores are already 1D
})

# Map topic labels and keywords
df_roberta_llm["kategoria_roberta_llm"] = df_roberta_llm["topic_id"].map(topic_to_label)
df_roberta_llm["Keywords"] = df_roberta_llm["topic_id"].map(topic_to_keywords)

# Convert keywords list to readable string
df_roberta_llm["Keywords"] = df_roberta_llm["Keywords"].apply(
    lambda x: ", ".join(x[:5]) if isinstance(x, list) else str(x)
)

# Add row index for reference
df_roberta_llm.reset_index(drop=True, inplace=True)
df_roberta_llm.insert(0, "row_id", range(len(df_roberta_llm)))

print("=== ENRICHED RESULTS DATAFRAME ===")
print(f"\nShape: {df_roberta_llm.shape}")
print(f"\nColumns: {df_roberta_llm.columns.tolist()}")
print(f"\nTopic distribution:")
print(df_roberta_llm["kategoria_roberta_llm"].value_counts())

print("\n--- Sample rows (first 10) ---")
print(df_roberta_llm[["row_id", "text_cleaned", "topic_id", "kategoria_roberta_llm", "confidence"]].head(10))

print("\n--- Sample rows by topic (one per topic) ---")
for topic_id in sorted(df_roberta_llm["topic_id"].unique()):
    sample = df_roberta_llm[df_roberta_llm["topic_id"] == topic_id].iloc[0]
    label = sample["kategoria_roberta_llm"]
    print(f"\nTopic {topic_id} ({label}):")
    print(f"  Text: {sample['text_cleaned'][:120]}...")
    print(f"  Keywords: {sample['Keywords']}")
    print(f"  Confidence: {sample['confidence']:.2%}")
    
df_roberta_llm.to_excel("output/enriched_results.xlsx", index=False)

In [ ]:
df_comparison = df_roberta_llm.join(df_allegro_reviews.set_index("text"), on="text", how="left")
df_comparison["Accuracy"] = df_comparison["kategoria_roberta_llm"] == df_comparison["kategoria_benchmark"]
df_comparison.to_excel("output/comparison_results.xlsx", index=False)

In [ ]:
# sample_texts = texts[:100] 
sample_texts = texts
business_categories = [
    "Słabe materiały i usterki", "Kompatybilność i montaż", 
    "Zawyżone parametry i podróbki", "Niska wydajność w praktyce", 
    "Logistyka i dostawa", "Obsługa klienta", 
    "Pozytywna ocena funkcjonalności", "Szum informacyjny"
]

# Initialize orchestrator with model selection
orchestrator = LLMOrchestrator(mode="gemini", gemini_model="gemini-2.5-flash")
pure_llm_results = []

# Zmienne do logowania całkowitego zużycia
total_run_tokens = 0

print(f"Rozpoczynam klasyfikację dla {len(sample_texts)} opinii (ze śledzeniem tokenów)...")

for text in tqdm(sample_texts, desc="Przetwarzanie API"):
        

    prompt = f"""Na podstawie poniższych słów kluczowych oraz przykładowych opinii klientów, przypisz tę grupę do DOKŁADNIE JEDNEJ z poniższych kategorii biznesowych:
    {', '.join(business_categories)}
    Opinia: "{text}"
    Zwróć wynik WYŁĄCZNIE jako obiekt JSON: {{"kategoria": "wybrana", "pewnosc": 95, "uzasadnienie": "..."}}"""
    
    try:
        # Używamy nowej metody zwracającej krotkę (tekst, statystyki)
        raw_response, token_stats = orchestrator._call_gemini_with_usage(prompt)
        
        # Dodajemy do ogólnego licznika
        total_run_tokens += token_stats["total_tokens"]
        
        raw_response = raw_response.strip()
        if raw_response.startswith("```json"):
            raw_response = raw_response[7:-3].strip()
        elif raw_response.startswith("```"):
            raw_response = raw_response[3:-3].strip()
            
        structured_data = json.loads(raw_response)
        
        if structured_data.get("kategoria") not in business_categories:
            structured_data["kategoria"] = "Inne"
            
        structured_data["tekst"] = text
        # LOGOWANIE TOKENÓW DLA TEGO KONKRETNEGO WIERSZA
        structured_data["prompt_tokens"] = token_stats["prompt_tokens"]
        structured_data["completion_tokens"] = token_stats["completion_tokens"]
        structured_data["total_tokens"] = token_stats["total_tokens"]
        
        pure_llm_results.append(structured_data)
             
    except Exception as e:
        pure_llm_results.append({
            "tekst": text, "kategoria": "Błąd API", "pewnosc": 0, 
            "uzasadnienie": str(e),
            "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0
        })

df_pure_llm = pd.DataFrame(pure_llm_results)

# WYŚWIETLANIE LOGÓW I KOSZTÓW
print("\n=== PODSUMOWANIE KOSZTÓW I ZUŻYCIA ===")
print(f"Łączna liczba zużytych tokenów: {total_run_tokens}")
# Przykładowy cennik Gemini Flash (np. $0.075 / 1 mln tokenów wejściowych, upraszczamy do średniej)
szacowany_koszt_usd = (total_run_tokens / 1_000_000) * 0.075 
print(f"Szacowany koszt dla tej próbki: ${szacowany_koszt_usd:.6f}")

print("\nPodgląd tabeli z logami tokenów:")
display(df_pure_llm[["kategoria", "pewnosc", "total_tokens", "uzasadnienie"]].head())
df_pure_llm.to_excel("output/pure_llm_classification_results.xlsx", index=False)
df_pure_llm

In [ ]:
# Porównanie wyników: BERTopic + Gemini vs Pure LLM
# Wiersze są w tej samej kolejności, więc łączymy po pozycji
df_comparison_pure_llm = pd.concat([
    df_pure_llm.reset_index(drop=True),
    df_allegro_reviews[["kategoria_benchmark"]].reset_index(drop=True)
], axis=1)

# Pomiń wiersze gdzie kategoria lub kategoria_benchmark są puste
df_comparison_pure_llm = df_comparison_pure_llm[
    (df_comparison_pure_llm["kategoria"].notna()) & 
    (df_comparison_pure_llm["kategoria"] != "") &
    (df_comparison_pure_llm["kategoria_benchmark"].notna()) & 
    (df_comparison_pure_llm["kategoria_benchmark"] != "")
]

df_comparison_pure_llm["match"] = df_comparison_pure_llm["kategoria"] == df_comparison_pure_llm["kategoria_benchmark"]
df_comparison_pure_llm["accuracy"] = df_comparison_pure_llm["match"].astype(int)

print(f"Accuracy: {df_comparison_pure_llm['accuracy'].mean():.2%}")
print(f"Matches: {df_comparison_pure_llm['match'].sum()} / {len(df_comparison_pure_llm)}")

df_comparison_pure_llm.to_excel("output/pure_llm_comparison_results.xlsx", index=False)
print("Zapisano porównanie do: output/pure_llm_comparison_results.xlsx")

In [ ]:
print(f"Rozpoczynam analizę NMF dla {len(texts)} opinii...")

# 1. Używamy TF-IDF zamiast CountVectorizer. 
# TF-IDF karze słowa, które występują wszędzie, a promuje te unikalne dla danego problemu.
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words=all_stopwords, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

# 2. INICJALIZACJA NMF (Redukujemy do 8 twardych kategorii biznesowych!)
num_topics = 50
nmf_model = NMF(n_components=num_topics, random_state=42, init='nndsvd')

# Trening modelu
nmf_model.fit(tfidf_matrix)

# 3. PREZENTACJA WYNIKÓW
def display_topics(model, feature_names, no_top_words):
    topics_dict = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
        topics_dict[f"Temat {topic_idx}"] = ", ".join(top_words)
        print(f"Kategoria {topic_idx}: {', '.join(top_words)}")
    return topics_dict

print("\n--- Najpopularniejsze słowa w klastrach (Wynik NMF) ---")
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
nmf_topics = display_topics(nmf_model, tfidf_feature_names, no_top_words=5)

# 4. PRZYPISANIE OPINII DO TEMATÓW
topic_results = nmf_model.transform(tfidf_matrix)
df_baseline = pd.DataFrame({
    "tekst": texts,
    "nmf_topic_id": topic_results.argmax(axis=1)
})

print("\nRozkład opinii w modelu NMF:")
print(df_baseline["nmf_topic_id"].value_counts())

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print(f"Rozpoczynam analizę NMF dla {len(texts)} opinii...")

# 1. TUNING TF-IDF: Ostrzejsze filtry częstotliwości
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=3, stop_words=all_stopwords, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

# 2. TUNING NMF: Dokładnie 8 klastrów, żeby pasowało do taksonomii biznesowej
num_topics = 8
nmf_model = NMF(n_components=num_topics, random_state=42, init='nndsvd')

# Trening modelu
nmf_model.fit(tfidf_matrix)

# 3. PREZENTACJA WYNIKÓW
def display_topics(model, feature_names, no_top_words):
    topics_dict = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]
        topics_dict[f"Temat {topic_idx}"] = ", ".join(top_words)
        print(f"Kategoria klasyczna {topic_idx}: {', '.join(top_words)}")
    return topics_dict

print("\n--- Najpopularniejsze słowa w klastrach (Zoptymalizowany NMF) ---")
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
nmf_topics = display_topics(nmf_model, tfidf_feature_names, no_top_words=6)

# 4. PRZYPISANIE OPINII DO TEMATÓW
topic_results = nmf_model.transform(tfidf_matrix)
df_baseline = pd.DataFrame({
    "tekst": texts,
    "nmf_topic_id": topic_results.argmax(axis=1)
})

print("\nRozkład opinii w modelu NMF (Baseline):")
print(df_baseline["nmf_topic_id"].value_counts())

In [ ]:
df_dev = pd.read_csv('data/downloaded/dev.tsv', sep='\t')

print(f"Rozkład wszystkich ocen w oryginalnym dev.tsv:\n{df_dev['rating'].value_counts().sort_index()}")

# 2. Stratified Sampling
# Pobieramy równo po 60 opinii dla każdej z 5 ocen (1.0, 2.0, 3.0, 4.0, 5.0) -> Razem 300
df_sample = df_dev.groupby('rating').sample(n=60, random_state=42)

# 3. Przetasowanie wierszy
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Przygotowanie struktury pliku dla Master LLM
df_sample['kategoria_benchmark'] = ""
df_sample['pewnosc'] = ""
df_sample['uzasadnienie'] = ""
df_sample['weryfikacja'] = "" # Tu wpiszesz 'OK' lub poprawisz kategorię
df_sample['sentyment'] = "" # Tu wpiszesz 'Pozytywny', 'Negatywny' lub 'Neutralny'

# 5. Zapis do Excela
output_file = 'output/ground_truth_sample.xlsx'
df_sample.to_excel(output_file, index=False)

print(f"\nUkończono! Zapisano zbalansowaną próbkę {len(df_sample)} opinii do pliku: {output_file}")
print("Podgląd wygenerowanego pliku:")
display(df_sample.head())

In [ ]:
# 1. INICJALIZACJA ORCHESTRATORA
# orchestrator = LLMOrchestrator(mode="gemini", gemini_model="gemini-3.1-pro-preview")
orchestrator = LLMOrchestrator(mode="gemini", gemini_model="gemini-3-flash-preview")

# 2. DEFINICJA PROMPTU SYSTEMOWEGO
prompt_template = """Jesteś Głównym Analitykiem Danych w firmie e-commerce. 
Twoim zadaniem jest stworzenie wzorcowego zbioru danych (Ground Truth) z recenzji sprzętu elektronicznego (uchwyty, powerbanki, szkła, słuchawki).

Masz do dyspozycji DOKŁADNIE 8 kategorii biznesowych:
1. "Słabe materiały i usterki" (pękający plastik, rysy, fizyczne zepsucie)
2. "Kompatybilność i montaż" (nie pasuje do telefonu/auta, odpada, za małe szkło)
3. "Zawyżone parametry i podróbki" (fałszywa pojemność baterii, brak certyfikatów, podróbki)
4. "Niska wydajność w praktyce" (działa, ale słabo: wolno ładuje, szum w słuchawkach, szarpie włosy)
5. "Logistyka i dostawa" (opóźnienia, zniszczona paczka, kurier)
6. "Obsługa klienta" (brak kontaktu, problemy ze zwrotem)
7. "Pozytywna ocena funkcjonalności" (klient chwali działanie i jakość)
8. "Szum informacyjny" (tylko oceny typu "ok", "polecam", "nie wiem", bez uzasadnienia)

ZASADY KRYTYCZNE:
- Wybierz kategorię, która stanowi GŁÓWNY powód napisania opinii.
- Zwróć wynik WYŁĄCZNIE jako poprawny obiekt JSON: 
{{"kategoria": "wybrana_nazwa_z_listy", "pewnosc": 90, "uzasadnienie": "maksymalnie jedno zdanie", "sentyment": "pozytywny"}}

Opinia do analizy: "{text}"
"""
# 3. LISTY NA WYNIKI
results = []
total_tokens_used = 0

print(f"Rozpoczynam generowanie Ground Truth dla {len(df_sample)} ankiet...")

# 4. GŁÓWNA PĘTLA PRZETWARZAJĄCA
for index, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Etykietowanie AI"):
    text_to_analyze = str(row['text'])
    prompt = prompt_template.format(text=text_to_analyze)
    
    try:
        # Wywołanie API
        raw_response, token_stats = orchestrator._call_gemini_with_usage(prompt)
        
        if token_stats:
            total_tokens_used += token_stats.get("total_tokens", 0)
        
        if raw_response:
            # Czyszczenie odpowiedzi z formatowania Markdown
            raw_response = raw_response.strip()
            if raw_response.startswith("```json"):
                raw_response = raw_response[7:-3].strip()
            elif raw_response.startswith("```"):
                raw_response = raw_response[3:-3].strip()
            
            # Parsowanie JSON
            structured_data = json.loads(raw_response)
            
            # Zapis wyników
            results.append({
                "kategoria_benchmark": structured_data.get("kategoria", "Błąd"),
                "pewnosc": structured_data.get("pewnosc", 0),
                "uzasadnienie": structured_data.get("uzasadnienie", ""),
                "sentyment": structured_data.get("sentyment", ""),
                "total_tokens": token_stats.get("total_tokens", 0) if token_stats else 0
            })
        else:
            raise ValueError("Pusta odpowiedź API")
            
    except Exception as e:
        results.append({
            "kategoria_benchmark": "Błąd API/Parsowania",
            "pewnosc": 0,
            "uzasadnienie": str(e),
            "sentyment": str(e),
            "total_tokens": 0
        })
        # Rate limiting prewencja
        time.sleep(2)

# 5. AKTUALIZACJA DATAFRAME I ZAPIS
df_results = pd.DataFrame(results)

# Połączenie oryginalnego DataFrame z nowymi kolumnami
for col in df_results.columns:
    df_sample[col] = df_results[col]

# Dodanie pustej kolumny dla weryfikacji manualnej
if 'weryfikacja' not in df_sample.columns:
    df_sample['weryfikacja'] = ""

output_file = 'output/ground_truth_master_filled.xlsx'
df_sample.to_excel(output_file, index=False)

print(f"\nZakończono! Zużyto łącznie {total_tokens_used} tokenów.")
print(f"Zapisano gotowy plik do: {output_file}")
display(df_sample[['rating', 'text', 'kategoria_benchmark', 'pewnosc']].head())